# Assignment 3 — Question 1

**Task:** For a real-world graph of at least 1000 nodes, implement:
- (a) Configuration Model
- (b) Edge-Swapping strategy

to generate random graphs that preserve the original degree sequence. Plot the original degree distribution alongside those generated by these strategies (averaged over 100 instances).

**Dataset:** Email-EU-core network (SNAP) — 1005 nodes, 25571 edges.

**Note on data:** The dataset (Email-EU-core) is downloaded automatically when you run the notebook. If the download fails, manually download the file from https://snap.stanford.edu/data/email-Eu-core.txt.gz and place it in the `Question 1/` folder alongside this notebook.

In [ ]:
import os
import gzip
import urllib.request
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

random.seed(42)
np.random.seed(42)

# Resolve this notebook's directory so all files are saved alongside it
try:
    SAVE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    SAVE_DIR = os.path.abspath(os.path.dirname(''))
print(f'Save directory: {SAVE_DIR}')

## 1. Download and Load Real-World Graph
We use the Email-EU-core network from SNAP (1005 nodes, 25571 edges).

In [ ]:
DATA_FILE = os.path.join(SAVE_DIR, 'email-Eu-core.txt.gz')
DATA_URL  = 'https://snap.stanford.edu/data/email-Eu-core.txt.gz'

if not os.path.exists(DATA_FILE):
    print('Downloading email-Eu-core dataset...')
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print('Download complete.')
else:
    print('Dataset already downloaded.')

G_real = nx.Graph()
with gzip.open(DATA_FILE, 'rt') as f:
    for line in f:
        line = line.strip()
        if line.startswith('#') or not line:
            continue
        u, v = map(int, line.split())
        if u != v:
            G_real.add_edge(u, v)

print(f'Nodes: {G_real.number_of_nodes()}, Edges: {G_real.number_of_edges()}')

degree_sequence = [d for _, d in G_real.degree()]
degree_counts   = Counter(degree_sequence)
k_vals          = sorted(degree_counts.keys())
n_nodes         = G_real.number_of_nodes()
p_k_orig        = [degree_counts[k] / n_nodes for k in k_vals]

print(f'Mean degree: {np.mean(degree_sequence):.2f}')
print(f'Max degree:  {max(degree_sequence)}')
print(f'Min degree:  {min(degree_sequence)}')

---
## Part (a) — Configuration Model

The Configuration Model generates a random graph by:
1. Creating *stubs* — each node gets as many stubs as its degree.
2. Randomly pairing stubs uniformly at random to form edges.
3. Self-loops and multi-edges are discarded (simple-graph result).

By construction, the expected degree sequence is preserved.

In [ ]:
def configuration_model(degree_seq):
    """Generate a random simple graph using the Configuration Model."""
    stubs = []
    for node, deg in enumerate(degree_seq):
        stubs.extend([node] * deg)
    if len(stubs) % 2 != 0:
        stubs.pop()
    random.shuffle(stubs)

    G = nx.Graph()
    G.add_nodes_from(range(len(degree_seq)))
    for i in range(0, len(stubs) - 1, 2):
        u, v = stubs[i], stubs[i + 1]
        if u != v:        # discard self-loops; nx.Graph drops multi-edges
            G.add_edge(u, v)
    return G

In [ ]:
N_INSTANCES = 100
orig_degree_seq = [d for _, d in G_real.degree()]

cm_degree_counter = Counter()
print(f'Running {N_INSTANCES} Configuration Model instances...')
for i in range(N_INSTANCES):
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N_INSTANCES}')
    G_cm = configuration_model(orig_degree_seq)
    for _, d in G_cm.degree():
        cm_degree_counter[d] += 1
print('Done.')

all_k_cm  = sorted(set(k_vals) | set(cm_degree_counter.keys()))
p_k_cm    = [cm_degree_counter.get(k, 0) / (N_INSTANCES * n_nodes) for k in all_k_cm]
p_k_real_cm = [degree_counts.get(k, 0) / n_nodes for k in all_k_cm]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(all_k_cm, p_k_real_cm, color='steelblue', alpha=0.6, width=1.0,
       label='Original network')
ax.bar(all_k_cm, p_k_cm, color='orange', alpha=0.6, width=1.0,
       label='Configuration Model (avg 100 instances)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree $k$', fontsize=12)
ax.set_ylabel('$P(k)$', fontsize=12)
ax.set_title('(a) Configuration Model — Degree Distribution\nvs. Original Network (Email-EU-core)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', linestyle='--', alpha=0.4)

plt.tight_layout()
out_path = os.path.join(SAVE_DIR, 'Q1a_configuration_model.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

### Discussion — Configuration Model

The averaged degree distribution of the Configuration Model closely follows the original network's distribution across all degree values. Minor deviations at high-degree (hub) nodes arise because the stub-pairing procedure occasionally produces self-loops and multi-edges that are discarded, slightly reducing the effective degree of high-degree nodes. Overall, the Configuration Model is a faithful null model for the degree sequence — it destroys higher-order structural properties (clustering, correlations) while preserving the degree distribution.

---
## Part (b) — Edge-Swapping Strategy

Starting from the original graph, repeatedly select two random edges (u, v) and (x, y) and swap them to (u, x) and (v, y). A swap is accepted only if it does not introduce self-loops or multi-edges. We perform Q × |E| successful swap attempts (Q = 10), which is the standard recommendation.

In [ ]:
def edge_swapping(G_original, Q=10):
    """Generate a random graph via Edge-Swapping; preserves the degree sequence exactly."""
    H = G_original.copy()
    target   = Q * H.number_of_edges()
    done     = 0
    attempts = 0
    max_att  = target * 10
    edges    = list(H.edges())

    while done < target and attempts < max_att:
        attempts += 1
        e1, e2 = random.sample(edges, 2)
        u, v   = e1
        x, y   = e2
        if random.random() < 0.5:
            x, y = y, x
        if (u != x and v != y and u != y and v != x
                and not H.has_edge(u, x) and not H.has_edge(v, y)):
            H.remove_edge(u, v)
            H.remove_edge(e2[0], e2[1])
            H.add_edge(u, x)
            H.add_edge(v, y)
            edges = list(H.edges())
            done += 1
    return H

In [ ]:
es_degree_counter = Counter()
print(f'Running {N_INSTANCES} Edge-Swapping instances...')
for i in range(N_INSTANCES):
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N_INSTANCES}')
    G_es = edge_swapping(G_real, Q=10)
    for _, d in G_es.degree():
        es_degree_counter[d] += 1
print('Done.')

all_k_es    = sorted(set(k_vals) | set(es_degree_counter.keys()))
p_k_es      = [es_degree_counter.get(k, 0) / (N_INSTANCES * n_nodes) for k in all_k_es]
p_k_real_es = [degree_counts.get(k, 0) / n_nodes for k in all_k_es]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(all_k_es, p_k_real_es, color='steelblue', alpha=0.6, width=1.0,
       label='Original network')
ax.bar(all_k_es, p_k_es, color='green', alpha=0.6, width=1.0,
       label='Edge Swapping (avg 100 instances)')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree $k$', fontsize=12)
ax.set_ylabel('$P(k)$', fontsize=12)
ax.set_title('(b) Edge-Swapping — Degree Distribution\nvs. Original Network (Email-EU-core)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', linestyle='--', alpha=0.4)

plt.tight_layout()
out_path = os.path.join(SAVE_DIR, 'Q1b_edge_swapping.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

### Discussion — Edge-Swapping

Because edge swaps only re-route edges without ever changing which nodes they attach to, the degree sequence is preserved **exactly** in every instance. The averaged degree distribution therefore matches the original perfectly — the two bars are identical. Unlike the Configuration Model, Edge-Swapping does not discard any edges, so there is no high-degree deficit. Both methods serve as degree-preserving null models, but Edge-Swapping gives a stricter guarantee on degree preservation while still effectively randomising the global wiring pattern.